##Climate Trace Dataset Consolidation

Source: https://huggingface.co/datasets/tjhunter/climate-trace/tree/main/v3-2024-ct5

In [13]:
# Dependencies

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, max, min, input_file_name
from pyspark.sql.types import DoubleType, StringType, IntegerType

# Create a SparkSession
spark = SparkSession.builder.appName("ClimateTrace").getOrCreate()

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# Parquet Exploration
import os

# Base directory containing year folders (2021-2024)
base_dir = "/content/drive/MyDrive/School Projects/Climate Trace Analysis/cleaned_data"

parquet_files = []

# Loop through year directories (2021-2024)
for year in range(2021, 2025):
    year_path = os.path.join(base_dir, str(year))

    for root, _, files in os.walk(year_path):
        for file in files:
            if file.endswith(".parquet"):
                parquet_files.append(os.path.join(root, file))  # Save full file path

# Print collected Parquet file paths
print("Parquet file paths:\n")
for path in parquet_files:
    print(path)

Parquet file paths:

/content/drive/MyDrive/School Projects/Climate Trace Analysis/cleaned_data/2021/cleaned_2021_n2o.parquet/part-00000-72ebe2bb-411c-4db7-b284-33b29e3b6c1e-c000.snappy.parquet
/content/drive/MyDrive/School Projects/Climate Trace Analysis/cleaned_data/2021/cleaned_2021_n2o.parquet/part-00001-72ebe2bb-411c-4db7-b284-33b29e3b6c1e-c000.snappy.parquet
/content/drive/MyDrive/School Projects/Climate Trace Analysis/cleaned_data/2021/cleaned_2021_co2e_100yr.parquet/part-00000-bec7ff9e-74a8-45f4-b6b3-8187d522ffe3-c000.snappy.parquet
/content/drive/MyDrive/School Projects/Climate Trace Analysis/cleaned_data/2021/cleaned_2021_co2e_100yr.parquet/part-00001-bec7ff9e-74a8-45f4-b6b3-8187d522ffe3-c000.snappy.parquet
/content/drive/MyDrive/School Projects/Climate Trace Analysis/cleaned_data/2021/cleaned_2021_ch4.parquet/part-00000-da714e08-27fe-404e-be90-f0ea04b491ea-c000.snappy.parquet
/content/drive/MyDrive/School Projects/Climate Trace Analysis/cleaned_data/2021/cleaned_2021_ch4.par

In [16]:
# Parquet Loading and Merge
for i, path in enumerate(parquet_files):
    if i == 0:
        df = spark.read.parquet(path)
    else:
        df = df.union(spark.read.parquet(path))

df.printSchema()

root
 |-- country_code: string (nullable = true)
 |-- source_id: integer (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- gas_type: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor_ton: double (nullable = true)
 |-- capacity_ha: double (nullable = true)
 |-- capacity_factor: double (nullable = true)
 |-- activity_ha: double (nullable = true)
 |-- created_date: date (nullable = true)
 |-- source_type: string (nullable = true)
 |-- confidence_source_type: string (nullable = true)
 |-- confidence_capacity: string (nullable = true)
 |-- confidence_capacity_factor: string (nullable = true)
 |-- confidence_activity: string (nullable = true)
 |-- confidence_emissions_factor: string (nullable = true)
 |-- confidence_emissions_quantity: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- latitude: double

In [ ]:
# Data Validation
# Get column names
column_names = [col.name for col in df.schema]

# Loop through each column
for column_name in column_names:
    print("=" * 50)
    print(f"🔎 Analyzing Column: {column_name}")
    print("=" * 50)

    # Data Type
    data_type = df.schema[column_name].dataType
    print(f"📌 Data Type: {data_type}")

    # Head Values
    print(f"\n📊 Top 5 Values:")
    df.select(column_name).show(5)

    # Tail Values (Sorted Descending)
    print(f"\n📉 Bottom 5 Values:")
    df.select(column_name).orderBy(col(column_name), ascending=False).show(5)

    # Min & Max for Numeric Data
    if isinstance(data_type, (IntegerType, DoubleType)):
        print(f"\n📈 Min & Max Values:")
        df.agg(
            max(col(column_name)).alias("Max Value"),
            min(col(column_name)).alias("Min Value")
        ).show()

    # Distinct Values for String Data
    if isinstance(data_type, (StringType)):
        print(f"\n🔹 Unique Values:")
        df.select(column_name).distinct().show(5)

        print(f"🔢 Unique Count: {df.select(column_name).distinct().count()}")

    # Total Count
    print(f"\n📊 Total Row Count:")
    print(df.select(column_name).count())

    # Value Counts (Frequency Distribution)
    print(f"\n📊 Value Distribution:")
    df.groupBy(column_name).count().orderBy(col("count"), ascending=False).show(5)

    # Null Count
    print(f"\n⚠️ Null Count: {df.filter(col(column_name).isNull()).count()}")

🔎 Analyzing Column: country_code
📌 Data Type: StringType()

📊 Top 5 Values:
+------------+
|country_code|
+------------+
|         ARE|
|         ARE|
|         ARE|
|         ARE|
|         ARE|
+------------+
only showing top 5 rows


📉 Bottom 5 Values:
+------------+
|country_code|
+------------+
|         ZWE|
|         ZWE|
|         ZWE|
|         ZWE|
|         ZWE|
+------------+
only showing top 5 rows


🔹 Unique Values:
+------------+
|country_code|
+------------+
|         NIU|
|         CCK|
|         HTI|
|         PSE|
|         BRB|
+------------+
only showing top 5 rows

🔢 Unique Count: 251

📊 Total Row Count:
242952000

📊 Value Distribution:


In [17]:
# Data Export
parquet_file = f"/content/drive/MyDrive/School Projects/Climate Trace Analysis/consolidated_cleaned_dataset"
df.coalesce(1).write.mode("overwrite").parquet(parquet_file)
print("Parquet file exported successfully.")

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 